# Bước 1: Khai báo thư viện và Đọc dữ liệu
Đọc file dữ liệu bổ sung từ thư mục `data/clean/` và loại bỏ các dòng trống rác để giữ nguyên 2.170 dòng thực tế.

In [1]:
import pandas as pd

# 1. Đọc dữ liệu thô
df = pd.read_csv("../data/clean/VNU_co_mo_phong_bo_sung.csv")

# 2. Lọc bỏ dòng trống hoàn toàn dựa trên cột định danh (STT)
df = df.dropna(subset=['STT'])

print(f"Số dòng chuẩn sau khi lọc: {len(df)}")

Số dòng chuẩn sau khi lọc: 2770


# Bước 2: Đồng bộ tên cột và Đánh dấu dữ liệu
- Đổi tên các cột bị sai chính tả theo đúng chuẩn từ điển dữ liệu.
- Đánh dấu 226 dòng trùng lặp (dùng cột `Trung_lap`, bỏ qua cột `STT` khi so sánh) và đánh dấu 442 dòng trả lời đều (dùng cột `Tra_loi_deu`). **Tuyệt đối không xóa dữ liệu gốc.**

In [2]:
# 1. Đổi tên 3 cột sai chính tả
df = df.rename(columns={
    'Time_SocicalMedia': 'Time_SocialMedia',
    'Facilitie_Uni': 'Facilities_Uni',
    'InfuenceF_Friends': 'Influence_Friends'
})

# 2. Xóa các dòng trùng lặp (Loại bỏ double-click/spam)
cot_kiem_tra = df.columns.drop('STT')
so_dong_ban_dau = len(df)
df = df.drop_duplicates(subset=cot_kiem_tra, keep='first')
so_dong_da_xoa = so_dong_ban_dau - len(df)
print(f"Đã xóa {so_dong_da_xoa} dòng trùng lặp.")
print(f"Số dòng thực tế còn lại: {len(df)}")

# 3. Đánh dấu dòng trả lời đều (chọn 1 mức cho cả 9 câu Likert)
likert_cols = [
    'Adapt_Learning_Uni', 'Study_Methods', 'SupportOf_Uni', 
    'SupportOf_Lec', 'Facilities_Uni', 'Quality_Lecturer', 
    'TrainingCurriculum', 'Competitive_Class', 'Influence_Friends'
]
df['Tra_loi_deu'] = df[likert_cols].nunique(axis=1) == 1
print(f"Số dòng trả lời đều (bừa): {df['Tra_loi_deu'].sum()}")

Đã xóa 231 dòng trùng lặp.
Số dòng thực tế còn lại: 2539
Số dòng trả lời đều (bừa): 461


# Bước 3: Giải mã dữ liệu phân loại (Mapping)
Sử dụng Codebook để chuyển đổi các giá trị số thành nhãn văn bản tiếng Việt dễ đọc, phục vụ cho việc làm Dashboard sau này.

In [3]:
# BƯỚC 3: GIẢI MÃ TOÀN BỘ 22 CỘT TỪ SỐ SANG CHỮ

# 1. Các biến cơ bản (Năm học, Giới tính, Có/Không)
df['Year'] = df['Year'].map({1: 'Năm nhất', 2: 'Năm hai', 3: 'Năm ba', 4: 'Năm tư', 5: 'Đã tốt nghiệp'})
df['Gender'] = df['Gender'].map({1: 'Nam', 2: 'Nữ'})

dict_co_khong = {1: 'Có', 2: 'Không'}
df['Policy_Stu'] = df['Policy_Stu'].map(dict_co_khong)
df['Minority_Stu'] = df['Minority_Stu'].map(dict_co_khong)
df['Poor_Stu'] = df['Poor_Stu'].map(dict_co_khong)

# 2. Các biến Giáo dục & Nghề nghiệp của Phụ huynh
dict_hoc_van = {
    1: 'Tiểu học', 2: 'THCS', 3: 'THPT', 4: 'Cao đẳng', 
    5: 'Đại học/Sau đại học', 6: 'Khác'
}
df['Father_Edu'] = df['Father_Edu'].map(dict_hoc_van)
df['Mother_Edu'] = df['Mother_Edu'].map(dict_hoc_van)

dict_nghe_nghiep = {
    1: 'Cán bộ nhà nước', 2: 'Tự kinh doanh', 3: 'Làm việc tự do', 
    4: 'Khác', 5: 'Không tiết lộ'
}
df['Father_Occupation'] = df['Father_Occupation'].map(dict_nghe_nghiep)
df['Mother_Occupation'] = df['Mother_Occupation'].map(dict_nghe_nghiep)

# 3. Các biến Thời gian (Bạn bè, Mạng xã hội, Tự học)
# Lưu ý: Cột Time_SocialMedia đã được sửa đúng chính tả ở Bước 2
dict_thoi_gian_1_4 = {
    1: 'Dưới 1 giờ', 2: 'Từ 1 đến dưới 2 giờ', 
    3: 'Từ 2 đến dưới 3 giờ', 4: 'Từ 3 đến dưới 4 giờ', 5: 'Hơn 4 giờ'
}
df['Time_Friends'] = df['Time_Friends'].map(dict_thoi_gian_1_4)
df['Time_SocialMedia'] = df['Time_SocialMedia'].map(dict_thoi_gian_1_4)

df['Time_Studying'] = df['Time_Studying'].map({
    1: 'Dưới 2 giờ', 2: 'Từ 2 đến dưới 4 giờ', 
    3: 'Từ 4 đến dưới 6 giờ', 4: 'Từ 6 đến dưới 8 giờ', 5: 'Hơn 8 giờ'
})

# 4. Biến Điểm trung bình (GPA)
df['GPA'] = df['GPA'].map({
    1: 'Kém (Dưới 2.0)', 
    2: 'Trung bình (2.0 - 2.5)', 
    3: 'Khá (2.5 - 3.2)', 
    4: 'Giỏi (3.2 - 3.6)', 
    5: 'Xuất sắc (Trên 3.6)'
})

# 5. Giải mã 9 câu Likert (Phần B)
# Thang đo 1-5 gốc: Not at all, Little, Moderate, Quite, Very
thang_do_likert = {
    1: 'Hoàn toàn không', 
    2: 'Ít', 
    3: 'Bình thường', 
    4: 'Nhiều', 
    5: 'Rất nhiều'
}

likert_cols = [
    'Adapt_Learning_Uni', 'Study_Methods', 'SupportOf_Uni', 
    'SupportOf_Lec', 'Facilities_Uni', 'Quality_Lecturer', 
    'TrainingCurriculum', 'Competitive_Class', 'Influence_Friends'
]

# Vòng lặp map dữ liệu cho 9 cột Likert
for col in likert_cols:
    df[col] = df[col].map(thang_do_likert)

print("Đã giải mã thành công toàn bộ 22 biến theo đúng Codebook gốc!")

Đã giải mã thành công toàn bộ 22 biến theo đúng Codebook gốc!


# Bước 4: Kiểm tra toàn vẹn và Xuất file
Kiểm tra lại tổng số dòng dữ liệu để đảm bảo không bị thất thoát, sau đó xuất ra file `vnu_that_sach.csv` mới để cả nhóm sử dụng làm báo cáo và Dashboard.

In [4]:
# Kiểm tra tổng số dòng cuối cùng
print(f"Tổng số dòng dữ liệu sạch cuối cùng: {len(df)}")

# BIÊN DỊCH TIÊU ĐỀ CỘT SANG TIẾNG VIỆT
dict_dich_ten_cot = {
    # Nhóm A: Thông tin cá nhân & Gia đình
    'Year': 'Năm học',
    'Gender': 'Giới tính',
    'Policy_Stu': 'Diện chính sách',
    'Minority_Stu': 'Dân tộc thiểu số',
    'Poor_Stu': 'Hộ nghèo',
    'Father_Edu': 'Học vấn của Bố',
    'Mother_Edu': 'Học vấn của Mẹ',
    'Father_Occupation': 'Nghề nghiệp của Bố',
    'Mother_Occupation': 'Nghề nghiệp của Mẹ',
    
    # Nhóm A: Thời gian
    'Time_Friends': 'TG dành cho bạn bè',
    'Time_SocialMedia': 'TG dùng mạng xã hội',
    'Time_Studying': 'TG tự học',
    'GPA': 'Điểm GPA',
    
    # Nhóm B: 9 câu hỏi thang đo Likert
    'Adapt_Learning_Uni': 'Mức độ thích nghi ĐH',
    'Study_Methods': 'Phương pháp học tập',
    'SupportOf_Uni': 'Sự hỗ trợ từ Trường',
    'SupportOf_Lec': 'Sự hỗ trợ từ Giảng viên',
    'Facilities_Uni': 'Cơ sở vật chất',
    'Quality_Lecturer': 'Chất lượng Giảng viên',
    'TrainingCurriculum': 'Chương trình đào tạo',
    'Competitive_Class': 'Sự cạnh tranh trong lớp',
    'Influence_Friends': 'Ảnh hưởng từ bạn bè'
}

# Tiến hành đổi tên cột
df = df.rename(columns=dict_dich_ten_cot)

print("Đã biên dịch toàn bộ tiêu đề cột sang tiếng Việt!")

# Xuất file kết quả vào thư mục data/clean/ (không lưu kèm cột index thừa)
df.to_csv("../data/clean/vnu_that_sach.csv", index=False)

print("Đã xuất file ../data/clean/vnu_that_sach.csv thành công!")

Tổng số dòng dữ liệu sạch cuối cùng: 2539
Đã biên dịch toàn bộ tiêu đề cột sang tiếng Việt!
Đã xuất file ../data/clean/vnu_that_sach.csv thành công!
